# DataLoader:


- should be able to handle: real data, categorical data.
- pytorch-forecasting: batching only by sequence length => lacks batching by time interval (e.g. "get all time steps within the last 60 minutes")

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np
from torch import Tensor
import pandas as pd
from pandas import DataFrame, Series, Timestamp, Timedelta, DatetimeIndex
from pandas.tseries.offsets import DateOffset
import warnings

pd.set_option('display.max_rows', 5)

In [3]:
from tsdm.datasets import Electricity

X = Electricity.dataset
X

client,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,MT_010,...,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369,MT_370
time,,,,,,,,,,,,,,,,,,,,,
2011-01-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2011-01-01 00:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-12-31 23:45:00,1.269036,21.337127,1.737619,166.666667,85.365854,285.714286,10.17524,225.589226,64.685315,72.043011,...,246.252677,28000.0,1443.037975,909.090909,26.075619,4.095963,664.618086,146.911519,646.627566,6540.540541
2015-01-01 00:00:00,2.538071,19.914651,1.737619,178.861789,84.146341,279.761905,10.17524,249.158249,62.937063,69.892473,...,188.436831,27800.0,1409.282700,954.545455,27.379400,4.095963,628.621598,131.886477,673.020528,7135.135135


In [5]:
from tsdm.util.converters import time2int, time2float
time2float(X.index), time2int(X.index)

(Float64Index([     0.0,      1.0,      2.0,      3.0,      4.0,      5.0,
                    6.0,      7.0,      8.0,      9.0,
               ...
               140246.0, 140247.0, 140248.0, 140249.0, 140250.0, 140251.0,
               140252.0, 140253.0, 140254.0, 140255.0],
              dtype='float64', name='time', length=140256),
 Int64Index([     0,      1,      2,      3,      4,      5,      6,      7,
                  8,      9,
             ...
             140246, 140247, 140248, 140249, 140250, 140251, 140252, 140253,
             140254, 140255],
            dtype='int64', name='time', length=140256))

In [6]:
from tsdm.util import time_gcd, is_quasiregular, is_regular, regularity_coefficient

print(F"{is_regular(X.index)=}")
print(F"{is_quasiregular(X.index)=}")
print(F"{regularity_coefficient(X.index)=}")
print(F"{time_gcd(X.index)=}")


is_regular(X.index)=True
is_quasiregular(X.index)=True
regularity_coefficient(X.index)=1.0
time_gcd(X.index)=numpy.timedelta64(900000000000,'ns')


In [7]:
static_categoricals: list[str] = [],
static_reals: list[str] = [], 
time_varying_known_categoricals: list[str] = [], 
time_varying_known_reals: list[str] = [], 
time_varying_unknown_categoricals: list[str] = [], 
time_varying_unknown_reals: list[str] = []

In [8]:
x = np.random.randint(low=-5, high=+5, size=(3,3,4,5))
mask = np.random.choice([False, True], x.shape)

In [9]:
ds = Series(x.flatten()).astype(pd.Int64Dtype())
ds = ds.where(mask.flatten())
ds = ds.astype('category')
ds

0      NaN
1        2
      ... 
178    NaN
179      1
Length: 180, dtype: category
Categories (10, object): [-5, -4, -3, -2, ..., 1, 2, 3, 4]

In [10]:
pd.get_dummies(ds, sparse=True, dtype=float)

,-5,-4,-3,-2,-1,0,1,2,3,4
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
178,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
179,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [11]:
pd.NA

<NA>

In [12]:
metadata : dict[str, Tensor]

In [15]:
from tsdm.util.converters import time2int, make_dense_triplets

df = make_dense_triplets(X).reset_index()
df

,time,variable,value
0,2011-01-01 00:15:00,MT_001,0.000000
1,2011-01-01 00:15:00,MT_002,0.000000
...,...,...,...
51894718,2015-01-01 00:00:00,MT_369,673.020528
51894719,2015-01-01 00:00:00,MT_370,7135.135135


In [16]:
split_dates = [Timestamp("2014-09-01"), Timestamp("2014-03-31"), X.index[-1]-DateOffset(days=7)]
assert Series(split_dates).isin(X.index).all()
split = split_dates[-1]

X_TRAIN = X.loc[:split].copy()
X_TEST  = X.loc[split:].copy()
X_TRAIN

client,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,MT_010,...,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369,MT_370
time,,,,,,,,,,,,,,,,,,,,,
2011-01-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2011-01-01 00:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-12-24 23:45:00,1.269036,20.625889,1.737619,162.601626,93.902439,297.619048,6.218202,262.626263,61.188811,59.139785,...,268.379729,16000.0,1611.814346,1000.000000,99.087353,4.681100,694.468832,116.861436,678.152493,6594.594595
2014-12-25 00:00:00,2.538071,19.914651,1.737619,160.569106,89.024390,261.904762,5.087620,269.360269,59.440559,60.215054,...,224.839400,16100.0,1426.160338,977.272727,95.176010,4.095963,698.858648,110.183639,694.281525,6810.810811


# Pre-processing

## Option 1: aggregation via sum /mean

In [17]:
X_TRAIN.resample('1H').sum()
X_TEST.resample('1H').mean()

client,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,MT_010,...,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369,MT_370
time,,,,,,,,,,,,,,,,,,,,,
2014-12-25 00:00:00,2.220812,19.914651,0.434405,151.930894,87.500000,268.601190,5.370266,257.575758,62.062937,60.215054,...,173.625981,13650.0,1165.611814,948.863636,44.328553,6.875366,661.325724,139.816361,687.316716,6635.135135
2014-12-25 01:00:00,1.586294,20.270270,0.000000,161.585366,87.804878,271.577381,5.652911,273.569024,59.440559,66.397849,...,79.942898,11375.0,781.645570,937.500000,26.075619,5.705091,601.404741,125.208681,677.236070,6540.540541
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-12-31 23:00:00,2.220812,21.337127,1.737619,161.585366,83.841463,308.035714,10.740531,250.841751,64.685315,72.580645,...,271.948608,28075.0,1546.413502,1232.954545,28.357236,7.314219,676.031607,161.519199,659.274194,6932.432432
2015-01-01 00:00:00,2.538071,19.914651,1.737619,178.861789,84.146341,279.761905,10.175240,249.158249,62.937063,69.892473,...,188.436831,27800.0,1409.282700,954.545455,27.379400,4.095963,628.621598,131.886477,673.020528,7135.135135


## Option 2: Normalization

In [18]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
X_TRAIN = X.loc[:split].copy()
X_TEST  = X.loc[split:].copy()

scaler = MinMaxScaler()
scaler.fit(X_TRAIN)
X_TRAIN.loc[:] = scaler.transform(X_TRAIN)
X_TEST.loc[:] = scaler.transform(X_TEST)
X_TRAIN

client,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,MT_010,...,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369,MT_370
time,,,,,,,,,,,,,,,,,,,,,
2011-01-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2011-01-01 00:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-12-24 23:45:00,0.026316,0.179012,0.011494,0.506329,0.626016,0.555556,0.139241,0.475610,0.388889,0.297297,...,0.314644,0.082988,0.207948,0.080734,0.295720,0.077670,0.609869,0.322581,0.437766,0.213287
2014-12-25 00:00:00,0.052632,0.172840,0.011494,0.500000,0.593496,0.488889,0.113924,0.487805,0.377778,0.302703,...,0.263598,0.083506,0.183996,0.078899,0.284047,0.067961,0.613724,0.304147,0.448178,0.220280


In [19]:
h_train = np.timedelta64(7, 'D')
h_pred = np.timedelta64(24, 'h')

lower = X.index[0]
upper = X.index[-1] - h_train - h_pred
mask = (X.index >= lower) & (X.index <= upper)
t_range = X.index[mask]
BATCHSIZE = 3

ini = np.random.choice(t_range)
mid = ini + h_train
end = ini + h_train + h_pred
train_mask = (X.index >= ini) & (X.index < mid)
train_time = X.index[train_mask]
train_data = X.loc[train_time]
valid_mask = (X.index >= mid) & (X.index < end)
valid_time = X.index[valid_mask]
valid_data = X.loc[valid_time]

In [20]:
class ContinuousTimeSeriesDataset(Dataset):


IndentationError: expected an indented block (1455694509.py, line 1)

In [ ]:
from pandas import DataFrame
from numpy import timedelta64
from scipy.stats import uniform
from typing import Union
from torch.utils.data import Dataset, DataLoader


class ContinuousTimeSeriesDataset(Dataset):
    def __init__(self, df: DataFrame, 
                 forecast_horizon: Union[timedelta64, int],
                 observed_horizon: Union[timedelta64, int],
                 dtype = torch.float32,
                 device = torch.device('cpu'),
                 pack_irregular_timeseries: bool=False):
        
        
        
        assert df.index.is_monotonic_increasing, "Index not sorted!" 
        if not df.index.is_monotonic_increasing:
            df = df.sort_index(ascending=True)
            
        self.κ = regularity_coefficient(df.index)
        self.gcd = time_gcd(df.index)
        self.is_regular = is_regular(df.index)
        
        print(F"Time Series is regular :  {self.is_regular}")
        print(F"Regularity Coefficient :  {self.κ}")
        print(F"Greatest Common Divisor:  {self.gcd}")

        
        assert np.issubdtype(df.index.dtype, np.datetime64), "This doesn't look"

        self.h_obs = observed_horizon
        self.h_pre = forecast_horizon
        self.time = df.index
        self.T = torch.tensor(time2float(df.index), dtype=dtype)
        self.X = torch.tensor(df.values, dtype=dtype)
        
#         self.lower = df.index[ 0] + observed_horizon
#         self.upper = df.index[-1] - forecast_horizon
        
#         assert self.lower <= self.upper, "The horizon is larger than the time range!"
        # uniform sampler on interval

    def __len__(self):
        return len(self.T)
    
    def __getitem__(self, idx):
        return self.T[idx], self.X[idx]
        
    

In [ ]:
DS = ContinuousTimeSeriesDataset(X, forecast_horizon=5, observed_horizon=5)

In [ ]:
train_loader = DataLoader(DS, batch_size=50)

In [ ]:
next(iter(train_loader))

In [ ]:
class FixedStepTimeSeriesDataset(Dataset):
    def __init__(self, df: DataFrame,  
                 forecast_horizon: timedelta64,
                 observed_horizon: timedelta64,
                 dtype = torch.float32,
                 device = torch.device('cpu')):
        
        
        assert np.issubdtype(df.index.dtype, np.datetime64), "This doesn't look"
        assert df.index.is_monotonic_increasing, "Index not sorted!" 
        if not df.index.is_monotonic_increasing
            df = df.sort_index(ascending=True)
        
        ΔT = np.diff(df.index)
        Δt = ΔT[0]
        assert np.all(ΔT == Δt), "Time Series irregular!"
        
        forecast_steps = timedelta64//
        observed_steps = 
        
        self.h_obs = observed_horizon
        self.h_pre = forecast_horizon
        self.time = pandas.Series(df.index)
        self.T = torch.tensor(time2float(df.index), dtype=dtype)
        self.X = torch.tensor(df.values, dtype=dtype)
        
        self.lower = df.index[ 0] + observed_horizon
        self.upper = df.index[-1] - forecast_horizon
        
        assert self.lower <= self.upper, "The horizon is larger than the time range!"

        min_index = T[T >= T.iloc[ 0] + observed_horizon].index.min()
        max_index = T[T <= T.iloc[-1] - forecast_horizon].index.max()
        
        self.time_range = T[(T >= T.iloc[ 0] + observed_horizon) & (T <= T.iloc[-1] - forecast_horizon)]
        
    def __len__(self):
        return len(self.time_range)
    
    def __getitem__(self, idx):
        idx = self.time_range.index[idx]
        
        
        return self.T[idx], self.X[idx]
        

In [ ]:
T = pandas.Series(df.index)
X = df.values

In [ ]:
forecast_horizon = np.timedelta64(125, 'h')

In [ ]:
T[(T >= T.iloc[ 0] + observed_horizon) & (T <= T.iloc[-1] - forecast_horizon)].index[0]

In [ ]:
T[T <= T.iloc[-1] - forecast_horizon].index.max()

In [ ]:
T.index

In [ ]:
T.diff(2)

In [ ]:
T[(T <= T.iloc[-1] - np.timedelta64(125, 'h') - np.timedelta64(125, 'h'))].index

In [ ]:
import pandas
pandas.Series(T)

In [ ]:
 (T >= T[ 0] + np.timedelta64(125, 'h')).argmax()

In [ ]:
T[500]

In [ ]:
(T < T[-1] - np.timedelta64(125, 'h'))[::].argmax()

In [ ]:
(T < T[-1] - np.timedelta64(125, 'h')).sum()

In [ ]:
T[139755]

In [ ]:
from torch.utils.data import Dataset, DataLoader




In [ ]:
class TimeseriesDataset(torch.utils.data.Dataset):   
    def __init__(self, df: DataFrame
                 forecast_horizon: timedelta64,
                 observed_horizon: timedelta64,
                ):

        assert np.issubdtype(df.index.dtype, np.datetime64), "This doesn't look like a time series"

        if not df.index.is_monotonic_increasing
            warnings.warn("Index not sorted! Sorting..." )
            df = df.sort_index(ascending=True)

        self.time = df.index
        self.T = torch.tensor(time2float(df.index), dtype=dtype)
        self.X = torch.tensor(df.values, dtype=dtype)
        
        self.h_obs = observed_horizon
        self.h_pre = forecast_horizon
        
        self.lower = df.index[ 0] + observed_horizon
        self.upper = df.index[-1] - forecast_horizon
        
        
        
        
    def __len__(self):
        return self.X.__len__() - (self.seq_len-1)

    def __getitem__(self, index):
        return (self.X[index:index+self.seq_len], self.y[index+self.seq_len-1])